# BLS Imbalanced Learning — Complete Reproducible Experiment Notebook
## Comparing BLS · WBLS · CS-BLS · DKWBLS · **Con-BLS** across 8 Datasets

**Fully self-contained — just hit Runtime → Run all**

| Step | What happens |
|------|-------------|
| 0–1 | Install packages, configure globals |
| 2 | Download all 8 datasets automatically |
| 3 | Implement BLS, WBLS, CS-BLS, DKWBLS from scratch |
| 4 | Implement Con-BLS (contrastive learning-enhanced BLS) |
| 5 | 5-fold stratified CV — every model × every dataset |
| 6 | Full ablation study with delta tables |
| 7–12 | 6 publication-quality PNG figures |
| 13 | Wilcoxon significance tests + full CSV export |
| 14 | ZIP all outputs → browser download |

**Runtime:** ~10–15 min CPU  ·  ~3–5 min GPU (enable T4 in Runtime → Change runtime type)

## 0 · Install dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "scikit-learn", "matplotlib", "seaborn",
    "numpy", "pandas", "tqdm", "requests", "torch"])
print("Packages ready.")

## 1 · Imports & global configuration

In [ ]:
import os, json, zipfile, warnings
from pathlib import Path
from collections import Counter

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from scipy.stats import wilcoxon

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.datasets        import load_breast_cancer, fetch_openml
from sklearn.preprocessing   import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics         import (roc_auc_score, f1_score,
                                     matthews_corrcoef, confusion_matrix)

warnings.filterwarnings("ignore")

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED    = 42
N_FOLDS = 5
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Output directory ──────────────────────────────────────────────────────────
OUT = Path("bls_outputs")
OUT.mkdir(exist_ok=True)

# ── Colour palette ─────────────────────────────────────────────────────────────
PAL = {
    "BLS":     "#4C72B0",
    "WBLS":    "#DD8452",
    "CS-BLS":  "#55A868",
    "DKWBLS":  "#C44E52",
    "Con-BLS": "#1D9E75",   # proposed method
}
MODEL_ORDER = ["BLS", "WBLS", "CS-BLS", "DKWBLS", "Con-BLS"]

# ── Publication plot style ─────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family"       : "DejaVu Sans",
    "font.size"         : 12,
    "axes.titlesize"    : 13,
    "axes.labelsize"    : 12,
    "xtick.labelsize"   : 10,
    "ytick.labelsize"   : 10,
    "legend.fontsize"   : 10,
    "figure.dpi"        : 150,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.alpha"        : 0.3,
    "grid.linewidth"    : 0.6,
})
print("Configuration ready.")

## 2 · Dataset download & preparation

In [ ]:
# ─── Helpers ──────────────────────────────────────────────────────────────────
def ir(y):
    c = Counter(y)
    return max(c.values()) / min(c.values())

def synth_fallback(n, d, frac, seed):
    rng = np.random.RandomState(seed)
    X   = rng.randn(n, d).astype(np.float32)
    y   = (rng.rand(n) < frac).astype(int)
    return X, y

# ─── Individual loaders ────────────────────────────────────────────────────────
def load_pima():
    url = ("https://raw.githubusercontent.com/jbrownlee/"
           "Datasets/master/pima-indians-diabetes.data.csv")
    try:
        df = pd.read_csv(url, header=None)
        return df.iloc[:,:-1].values.astype(np.float32), df.iloc[:,-1].values.astype(int)
    except Exception:
        return synth_fallback(768, 8, 0.349, 1)

def load_wdbc():
    d = load_breast_cancer()
    return d.data.astype(np.float32), d.target.astype(int)

def load_glass():
    try:
        d = fetch_openml("glass", version=1, as_frame=False, parser="auto")
        X = d.data.astype(np.float32)
        # class 1 (building_windows_float_processed) vs rest  — IR ≈ 3.2
        y = (d.target.astype(int) == 1).astype(int)
        return X, y
    except Exception:
        return synth_fallback(214, 9, 0.32, 2)

def load_ecoli():
    try:
        d = fetch_openml("ecoli", version=1, as_frame=False, parser="auto")
        X = d.data.astype(np.float32)
        y = np.array([1 if str(v).strip() in ("im","imS","imL","imU") else 0
                      for v in d.target], dtype=int)
        return X, y
    except Exception:
        return synth_fallback(336, 7, 0.029, 4)

def load_credit():
    try:
        d   = fetch_openml("creditcard", version=1, as_frame=False, parser="auto")
        X   = d.data.astype(np.float32)
        y   = d.target.astype(int)
        rng = np.random.RandomState(SEED)
        mi  = np.where(y==1)[0]; ma = np.where(y==0)[0]
        ms  = rng.choice(ma, 5000, replace=False)
        idx = np.concatenate([ms, mi]); rng.shuffle(idx)
        return X[idx], y[idx]
    except Exception:
        rng = np.random.RandomState(5)
        X   = np.vstack([rng.randn(5000,29), rng.randn(492,29)+2]).astype(np.float32)
        y   = np.array([0]*5000+[1]*492)
        p   = rng.permutation(len(y))
        return X[p], y[p]

def load_nslkdd():
    url = ("https://raw.githubusercontent.com/defcom17/"
           "NSL_KDD/master/KDDTrain+.txt")
    cols = [f"f{i}" for i in range(41)] + ["label","difficulty"]
    try:
        df  = pd.read_csv(url, header=None, names=cols)
        df  = pd.get_dummies(df, columns=["f1","f2","f3"])
        df["y"] = (df["label"] != "normal").astype(int)
        fc  = [c for c in df.columns if c not in ("label","difficulty","y")]
        X   = df[fc].values.astype(np.float32)
        y   = df["y"].values.astype(int)
        rng = np.random.RandomState(SEED)
        ni  = np.where(y==0)[0]; ai = np.where(y==1)[0]
        ns  = rng.choice(ni, min(8000,len(ni)), replace=False)
        idx = np.concatenate([ns,ai]); rng.shuffle(idx)
        return X[idx], y[idx]
    except Exception:
        rng = np.random.RandomState(6)
        X   = rng.randn(5000, 40).astype(np.float32)
        y   = (rng.rand(5000) < 0.48).astype(int)
        return X, y

def load_synthetic():
    rng = np.random.RandomState(SEED)
    X   = np.vstack([rng.randn(1950,10), rng.randn(50,10)+3]).astype(np.float32)
    y   = np.array([0]*1950+[1]*50)
    p   = rng.permutation(len(y))
    return X[p], y[p]

# ─── Load all ──────────────────────────────────────────────────────────────────
LOADERS = [
    ("Pima",       load_pima),
    ("WDBC",       load_wdbc),
    ("Glass",      load_glass),
    ("Ecoli",      load_ecoli),
    ("Credit",     load_credit),
    ("NSL-KDD",    load_nslkdd),
    ("Synth-IR50", load_synthetic),
]

DATASETS = {}
print(f"{'Dataset':<14} {'n':>7} {'d':>5} {'min%':>7}  {'IR':>7}")
print("-"*46)
for name, fn in LOADERS:
    X, y = fn()
    c    = Counter(y)
    DATASETS[name] = (X, y)
    print(f"{name:<14} {len(y):>7} {X.shape[1]:>5} "
          f"{100*c[1]/len(y):>6.1f}%  {ir(y):>6.1f}:1")
print(f"\nLoaded {len(DATASETS)} datasets.")

## 3 · BLS family implementations

## 4 · Con-BLS — Proposed contrastive learning-enhanced BLS

In [ ]:
# ─── Shared utilities ─────────────────────────────────────────────────────────
def inv_freq_weights(y):
    """Inverse-frequency per-sample weights (used by WBLS, CS-BLS, DKWBLS)."""
    cls, cnt = np.unique(y, return_counts=True)
    n = len(y); C = len(cls)
    w = {c: n / (C * k) for c, k in zip(cls, cnt)}
    return np.array([w[yi] for yi in y], dtype=np.float64)

def log_cost_weights(y):
    """Log-cost weights used by CS-BLS (Tao et al., 2022)."""
    cls, cnt = np.unique(y, return_counts=True)
    n = len(y)
    w = {c: np.log(n / k + 1) for c, k in zip(cls, cnt)}
    return np.array([w[yi] for yi in y], dtype=np.float64)

def solve_ridge(A, Y, lam, omega=None):
    """Analytical ridge regression.  Omega = diagonal weight matrix."""
    lI = lam * np.eye(A.shape[1])
    if omega is not None:
        Om = np.diag(omega)
        return np.linalg.solve(A.T @ Om @ A + lI, A.T @ Om @ Y)
    return np.linalg.solve(A.T @ A + lI, A.T @ Y)

# ─── Base BLS (Chen & Liu, IEEE TNNLS 2018) ───────────────────────────────────
class BLS:
    """
    Standard Broad Learning System.
    Architecture: Input → N feature-node groups → M enhancement-node groups
                  → Analytical output weights via ridge regression.
    """
    def __init__(self, n_fg=10, n_fp=10, n_eg=10, n_ep=10,
                 lam=1e-3, seed=SEED):
        self.n_fg = n_fg; self.n_fp = n_fp
        self.n_eg = n_eg; self.n_ep = n_ep
        self.lam  = lam;  self.seed = seed
        self.sc   = StandardScaler()

    # ── random feature nodes ──────────────────────────────────────────────────
    def _make_feat(self, X, fit=False):
        rng = np.random.RandomState(self.seed)
        d   = X.shape[1]
        if fit:
            self._Wf = [rng.randn(d, self.n_fp) * 0.3 for _ in range(self.n_fg)]
            self._bf = [rng.randn(1, self.n_fp) * 0.1 for _ in range(self.n_fg)]
        Zl = []
        for W, b in zip(self._Wf, self._bf):
            Z = np.maximum(0, X @ W + b)           # ReLU
            Z = (Z - Z.mean(0)) / (Z.std(0) + 1e-8)
            Zl.append(Z)
        return np.hstack(Zl)

    # ── random enhancement nodes ──────────────────────────────────────────────
    def _make_enh(self, Z, fit=False):
        rng = np.random.RandomState(self.seed + 999)
        d   = Z.shape[1]
        if fit:
            self._We = [rng.randn(d, self.n_ep) * 0.1 for _ in range(self.n_eg)]
            self._be = [rng.randn(1, self.n_ep) * 0.1 for _ in range(self.n_eg)]
        return np.hstack([np.tanh(Z @ W + b) for W, b in zip(self._We, self._be)])

    def _rep(self, X, fit=False):
        Z = self._make_feat(X, fit)
        return np.hstack([Z, self._make_enh(Z, fit)])

    # ── fit ───────────────────────────────────────────────────────────────────
    def fit(self, X, y, omega=None):
        Xs       = self.sc.fit_transform(X).astype(np.float32)
        nc       = len(np.unique(y))
        Y        = np.eye(nc)[y]
        A        = self._rep(Xs, fit=True)
        self.W   = solve_ridge(A, Y, self.lam, omega)
        self.nc_ = nc
        return self

    def predict_proba(self, X):
        Xs = self.sc.transform(X).astype(np.float32)
        A  = self._rep(Xs)
        lg = A @ self.W
        e  = np.exp(lg - lg.max(1, keepdims=True))
        return e / e.sum(1, keepdims=True)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), 1)

    def feat_embed(self, X):
        """Return feature-node representations (used for t-SNE)."""
        return self._make_feat(self.sc.transform(X).astype(np.float32))


# ─── WBLS — Weighted BLS (Yang et al., IEEE Trans. Serv. Comput., 2021) ───────
class WBLS(BLS):
    """
    Assigns inverse-frequency weights to minority samples so the
    ridge regression penalises minority misclassification more heavily.
    """
    def fit(self, X, y):
        return super().fit(X, y, omega=inv_freq_weights(y))


# ─── CS-BLS — Cost-Sensitive BLS (Tao et al., Mathematics, 2022) ──────────────
class CSBLS(BLS):
    """
    Uses log-cost weights: w_c = log(N/n_c + 1).
    Produces larger penalties for rarer classes than linear inverse-freq.
    """
    def fit(self, X, y):
        return super().fit(X, y, omega=log_cost_weights(y))


# ─── DKWBLS — Double-Kernel Weighted BLS (Chen et al., Neural Comput. Appl., 2022)
class DKWBLS(BLS):
    """
    Augments the input with Random Fourier Features that approximate an RBF
    kernel, then applies inverse-frequency cost weighting on the combined
    [original + kernel] feature space.
    """
    def __init__(self, gamma=1.0, n_rff=64, **kw):
        super().__init__(**kw)
        self.gamma = gamma
        self.n_rff = n_rff

    # ── Random Fourier Features (Rahimi & Recht, NeurIPS 2007) ───────────────
    def _rff(self, X, fit=False):
        rng = np.random.RandomState(self.seed + 2000)
        if fit:
            d          = X.shape[1]
            self._Wrff = rng.randn(d, self.n_rff) * np.sqrt(2 * self.gamma)
            self._brff = rng.uniform(0, 2 * np.pi, self.n_rff)
        return np.sqrt(2 / self.n_rff) * np.cos(X @ self._Wrff + self._brff)

    def fit(self, X, y):
        Xs       = self.sc.fit_transform(X).astype(np.float32)
        Xk       = self._rff(Xs, fit=True)
        Xa       = np.hstack([Xs, Xk])          # double-kernel augmented input
        nc       = len(np.unique(y))
        Y        = np.eye(nc)[y]
        A        = self._rep(Xa, fit=True)
        omega    = inv_freq_weights(y)
        self.W   = solve_ridge(A, Y, self.lam, omega)
        self.nc_ = nc
        self._Xa_fit = Xa                        # store for transform step
        return self

    def _predict_rep(self, X):
        Xs = self.sc.transform(X).astype(np.float32)
        Xk = self._rff(Xs)
        return self._rep(np.hstack([Xs, Xk]))

    def predict_proba(self, X):
        A  = self._predict_rep(X)
        lg = A @ self.W
        e  = np.exp(lg - lg.max(1, keepdims=True))
        return e / e.sum(1, keepdims=True)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), 1)

    def feat_embed(self, X):
        Xs = self.sc.transform(X).astype(np.float32)
        Xk = self._rff(Xs)
        return self._make_feat(np.hstack([Xs, Xk]))


print("BLS  WBLS  CS-BLS  DKWBLS — all ready.")

In [ ]:
# ─── Con-BLS — Contrastive Learning-Enhanced BLS (proposed method) ────────────
#
# Three-stage architecture:
#   Stage 1: Trainable feature nodes optimised with Supervised Contrastive Loss
#            + learnable class prototype anchors (Khosla et al., NeurIPS 2020)
#            + Minority-Aware Batch Sampling (MABS) to counter class imbalance
#   Stage 2: Random fixed enhancement nodes (standard BLS expansion)
#   Stage 3: Analytical output weights via cost-sensitive ridge regression
#
# Key design: the contrastive objective shapes the feature space so minority
# representations are well-separated BEFORE the analytical solver runs,
# addressing the root cause that MMSE-based ridge regression ignores geometry.

# ── 4a. Minority-Aware Batch Sampler ─────────────────────────────────────────
class MABSSampler:
    def __init__(self, X, y, bs=64):
        self.X = X; self.y = y; self.bs = bs
        self.cls = np.unique(y)
        self.idx = {c: np.where(y == c)[0] for c in self.cls}
        self.spc = max(1, bs // len(self.cls))

    def __iter__(self):
        nb = max(1, len(self.y) // self.bs)
        for _ in range(nb):
            bi = []
            for c in self.cls:
                bi.extend(np.random.choice(self.idx[c], self.spc, replace=True))
            bi = np.array(bi); np.random.shuffle(bi)
            yield self.X[bi], self.y[bi]

    def __len__(self):
        return max(1, len(self.y) // self.bs)


# ── 4b. Supervised Contrastive Loss (gradient-safe) ──────────────────────────
class SupConLoss(nn.Module):
    def __init__(self, tau=0.07):
        super().__init__(); self.tau = tau

    def forward(self, z, labels):
        z   = F.normalize(z, dim=1)
        N   = z.shape[0]; dev = z.device
        sim = torch.matmul(z, z.T) / self.tau
        lbl = labels.view(-1, 1)
        # positive-pair mask — all out-of-place (no masked_fill_ on live tensor)
        pos_mask = (lbl == lbl.T).float() * (1.0 - torch.eye(N, device=dev))
        # numerical stability shift (detached)
        sim = sim - sim.detach().max(dim=1, keepdim=True).values
        # zero diagonal via multiplication, not in-place
        exp_sim  = torch.exp(sim) * (1.0 - torch.eye(N, device=dev))
        log_denom = torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        log_prob  = sim - log_denom
        n_pos     = pos_mask.sum(dim=1)
        valid     = n_pos > 0
        if not valid.any():
            return torch.tensor(0.0, device=dev, requires_grad=True)
        per_anchor = -(pos_mask * log_prob).sum(dim=1) / n_pos.clamp(min=1.0)
        return per_anchor[valid].mean()


# ── 4c. Trainable feature-node network ───────────────────────────────────────
class FeatNet(nn.Module):
    def __init__(self, in_d, hid=128):
        super().__init__()
        self.enc  = nn.Sequential(
            nn.Linear(in_d, hid * 2), nn.BatchNorm1d(hid * 2), nn.ReLU(),
            nn.Linear(hid * 2, hid),  nn.BatchNorm1d(hid),     nn.ReLU())
        self.proj = nn.Sequential(
            nn.Linear(hid, hid), nn.ReLU(),
            nn.Linear(hid, hid // 2))
        self.hid = hid

    def forward(self, x):  return self.enc(x)
    def project(self, x):  return self.proj(self.enc(x))


# ── 4d. Con-BLS ───────────────────────────────────────────────────────────────
class ConBLS:
    def __init__(self, hid=128, n_eg=10, n_ep=10, lam=1e-3,
                 tau=0.07, lam1=0.5, lam2=0.1,
                 epochs=60, bs=64, lr=1e-3, seed=SEED):
        self.hid   = hid;  self.n_eg  = n_eg;  self.n_ep  = n_ep
        self.lam   = lam;  self.tau   = tau;   self.lam1  = lam1
        self.lam2  = lam2; self.epochs= epochs; self.bs   = bs
        self.lr    = lr;   self.seed  = seed
        self.sc      = StandardScaler()
        self.history = {"total": [], "con": [], "proto": []}

    # Stage 1: train feature nodes with contrastive objective
    def _train_feat(self, X, y):
        torch.manual_seed(self.seed)
        nc = len(np.unique(y)); d = X.shape[1]
        self.fn    = FeatNet(d, self.hid).to(DEVICE)
        self.proto = nn.Parameter(
            torch.randn(nc, self.hid // 2, device=DEVICE) * 0.01)
        opt   = torch.optim.Adam(
            list(self.fn.parameters()) + [self.proto],
            lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=self.epochs)
        scl = SupConLoss(self.tau)
        smp = MABSSampler(X, y, self.bs)
        self.nc_ = nc
        for _ in range(self.epochs):
            self.fn.train(); et = ec = ep = 0.0; nb = 0
            for Xb, yb in smp:
                Xt  = torch.FloatTensor(Xb).to(DEVICE)
                yt  = torch.LongTensor(yb).to(DEVICE)
                zp  = self.fn.project(Xt)
                lc  = scl(zp, yt)
                pn  = F.normalize(self.proto, dim=1)
                zn  = F.normalize(zp, dim=1)
                lpr = F.cross_entropy(torch.matmul(zn, pn.T) / self.tau, yt)
                loss = self.lam1 * lc + self.lam2 * lpr
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(
                    list(self.fn.parameters()) + [self.proto], 1.0)
                opt.step()
                et += loss.item(); ec += lc.item(); ep += lpr.item(); nb += 1
            sched.step()
            for k, v in zip(["total","con","proto"], [et, ec, ep]):
                self.history[k].append(v / max(1, nb))
        self.fn.eval()

    # Stage 2: random fixed enhancement nodes
    def _build_enh(self, Z, fit=False):
        rng = np.random.RandomState(self.seed + 999); d = Z.shape[1]
        if fit: self._We = []; self._be = []
        Hl = []
        for i in range(self.n_eg):
            if fit:
                self._We.append(rng.randn(d, self.n_ep) * 0.1)
                self._be.append(rng.randn(1, self.n_ep) * 0.1)
            Hl.append(np.tanh(Z @ self._We[i] + self._be[i]))
        return np.hstack(Hl)

    def _get_Z(self, X_np):
        self.fn.eval()
        with torch.no_grad():
            return self.fn(
                torch.FloatTensor(X_np).to(DEVICE)).cpu().numpy()

    # Stage 3: analytical output weights
    def fit(self, X, y):
        X  = self.sc.fit_transform(X).astype(np.float32)
        self._train_feat(X, y)
        Z  = self._get_Z(X)
        H  = self._build_enh(Z, fit=True)
        A  = np.hstack([Z, H])
        Y  = np.eye(self.nc_)[y]
        Om = np.diag(inv_freq_weights(y))
        lI = self.lam * np.eye(A.shape[1])
        self.W = np.linalg.solve(A.T @ Om @ A + lI, A.T @ Om @ Y)
        return self

    def predict_proba(self, X):
        X  = self.sc.transform(X).astype(np.float32)
        Z  = self._get_Z(X); H = self._build_enh(Z)
        A  = np.hstack([Z, H])
        lg = A @ self.W
        e  = np.exp(lg - lg.max(1, keepdims=True))
        return e / e.sum(1, keepdims=True)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), 1)

    def feat_embed(self, X):
        return self._get_Z(self.sc.transform(X).astype(np.float32))

print("Con-BLS ready (gradient-safe SupConLoss + MABS + prototype anchors).")

## 4 · Evaluation framework & 5-fold CV

In [ ]:
# ─── Metric helpers ───────────────────────────────────────────────────────────
def gmean(y_true, y_pred):
    cm  = confusion_matrix(y_true, y_pred)
    rec = cm.diagonal() / (cm.sum(1) + 1e-10)
    return float(np.prod(rec) ** (1 / len(rec)))

def all_metrics(model, X_te, y_te):
    """Return AUC, G-Mean, F1 (macro), MCC for a fitted model."""
    if len(np.unique(y_te)) < 2:
        return {k: float("nan") for k in ["AUC","GMean","F1","MCC"]}
    pr   = model.predict_proba(X_te)
    pred = model.predict(X_te)
    nc   = pr.shape[1]
    auc  = (roc_auc_score(y_te, pr[:,1]) if nc == 2
            else roc_auc_score(label_binarize(y_te,
                               classes=list(range(nc))),
                               pr, multi_class="ovr", average="macro"))
    return dict(
        AUC   = round(auc, 4),
        GMean = round(gmean(y_te, pred), 4),
        F1    = round(f1_score(y_te, pred, average="macro",
                               zero_division=0), 4),
        MCC   = round(matthews_corrcoef(y_te, pred), 4))

# ─── Model factory ────────────────────────────────────────────────────────────
def make_model(name):
    if name == "BLS":    return BLS()
    if name == "WBLS":   return WBLS()
    if name == "CS-BLS": return CSBLS()
    if name == "DKWBLS": return DKWBLS()
    if name == "Con-BLS": return ConBLS()
    raise ValueError(f"Unknown model: {name}")

# ─── Cross-validation runner ──────────────────────────────────────────────────
def cross_validate(name, X, y, n_splits=N_FOLDS):
    """
    Stratified k-fold CV.
    Skips folds where train or test has only one class (rare for high-IR data).
    Returns (mean_dict, std_dict).
    """
    skf   = StratifiedKFold(n_splits=n_splits, shuffle=True,
                             random_state=SEED)
    folds = []
    for fi, (tr, te) in enumerate(skf.split(X, y)):
        if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
            print(f"    [fold {fi+1}] skipped — single-class split")
            continue
        try:
            m = make_model(name)
            m.fit(X[tr], y[tr])
            folds.append(all_metrics(m, X[te], y[te]))
        except Exception as ex:
            print(f"    [fold {fi+1}] error: {ex.__class__.__name__}: {ex}")
    if not folds:
        empty = {k: float("nan") for k in ["AUC","GMean","F1","MCC"]}
        return empty, empty
    mn = {k: round(float(np.nanmean([r[k] for r in folds])), 4)
          for k in folds[0]}
    sd = {k: round(float(np.nanstd ([r[k] for r in folds])), 4)
          for k in folds[0]}
    return mn, sd

print("Evaluation framework ready.")

## 5 · Main experiments — 5-fold CV (all models × all datasets)

In [ ]:
RES = {}   # RES[dataset][model] = mean-metrics dict
STD = {}   # STD[dataset][model] = std-metrics dict

for ds_name, (X, y) in DATASETS.items():
    RES[ds_name] = {}; STD[ds_name] = {}
    print(f"\n{'='*54}")
    print(f"  {ds_name}   n={len(y)}  IR={ir(y):.1f}:1")
    print("="*54)
    for mname in MODEL_ORDER:
        print(f"  [{mname:<8}]  ", end="", flush=True)
        mn, sd = cross_validate(mname, X, y)
        RES[ds_name][mname] = mn
        STD[ds_name][mname] = sd
        print(f"AUC={mn['AUC']:.4f}  "
              f"GMean={mn['GMean']:.4f}  "
              f"F1={mn['F1']:.4f}  "
              f"MCC={mn['MCC']:.4f}")

print("\n\nAll experiments complete!")

## 6 · Results tables (console)

In [ ]:
METRICS = ["AUC","GMean","F1","MCC"]

def results_table(metric, show_std=True):
    ds_list = list(RES.keys()); cw = 19
    header  = f"{'Dataset':<14}" + "".join(f"{m:>{cw}}" for m in MODEL_ORDER)
    sep     = "-" * len(header)
    print(f"\n{sep}\nMetric : {metric}\n{sep}\n{header}\n{sep}")
    for ds in ds_list:
        vals = [RES[ds][m][metric] for m in MODEL_ORDER]
        best = max(v for v in vals if not np.isnan(v))
        row  = f"{ds:<14}"
        for m, v in zip(MODEL_ORDER, vals):
            sd   = STD[ds][m][metric]
            cell = f"{v:.4f}±{sd:.4f}" if show_std else f"{v:.4f}"
            mark = "*" if v == best else " "
            row += f"{(cell+mark):>{cw}}"
        print(row)
    print(sep + "\n  * = best per row")

for met in METRICS:
    results_table(met)

## 7 · Full ablation study

Compares all four BLS variants head-to-head on every dataset and metric.
The delta table shows how much each method improves over the plain BLS baseline.

In [ ]:
def ablation_table(metric):
    ds_list = list(RES.keys()); cw = 19
    header  = f"{'Dataset':<14}" + "".join(f"{m:>{cw}}" for m in MODEL_ORDER)
    sep     = "-" * len(header)
    print(f"\n{sep}\nAblation — {metric}\n{sep}\n{header}\n{sep}")
    for ds in ds_list:
        vals = [RES[ds][m][metric] for m in MODEL_ORDER]
        best = max(v for v in vals if not np.isnan(v))
        row  = f"{ds:<14}"
        for m, v in zip(MODEL_ORDER, vals):
            sd   = STD[ds][m][metric]
            cell = f"{v:.4f}±{sd:.4f}"
            mark = "*" if v == best else " "
            row += f"{(cell+mark):>{cw}}"
        print(row)
    print(sep + "\n  * = best per row")

for met in METRICS:
    ablation_table(met)

# ─── Delta over BLS baseline ──────────────────────────────────────────────────
print("\n" + "="*60)
print("Delta over BLS baseline (mean across all datasets)")
print("="*60)
print(f"{'Method':<10}", end="")
for met in METRICS: print(f"{met:>12}", end="")
print()
print("-"*58)
for m in MODEL_ORDER[1:]:          # skip BLS itself
    print(f"{m:<10}", end="")
    for met in METRICS:
        deltas = [RES[ds][m][met] - RES[ds]["BLS"][met]
                  for ds in DATASETS
                  if not np.isnan(RES[ds][m][met])
                  and not np.isnan(RES[ds]["BLS"][met])]
        print(f"{np.mean(deltas):>+12.4f}", end="")
    print()

# ─── Rank table ───────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("Average rank per model (1 = best) across all datasets")
print("="*60)
ds_list = list(RES.keys())
for met in METRICS:
    ranks = {m: [] for m in MODEL_ORDER}
    for ds in ds_list:
        vals  = {m: RES[ds][m][met] for m in MODEL_ORDER}
        order = sorted(MODEL_ORDER, key=lambda m: vals[m], reverse=True)
        for rank_i, m in enumerate(order, 1):
            ranks[m].append(rank_i)
    print(f"\n  {met}:  " +
          "  ".join(f"{m}={np.mean(ranks[m]):.2f}" for m in MODEL_ORDER))

## 8 · Figure 1 — AUC & G-Mean grouped bar charts

In [ ]:
def fig_grouped_bars(metric, fname, figsize=(15,5)):
    ds_list = list(RES.keys())
    x   = np.arange(len(ds_list))
    W   = 0.18; n = len(MODEL_ORDER)
    fig, ax = plt.subplots(figsize=figsize)

    for i, m in enumerate(MODEL_ORDER):
        vals = [RES[ds][m][metric] for ds in ds_list]
        errs = [STD[ds][m][metric] for ds in ds_list]
        off  = (i - n/2 + 0.5) * W
        ax.bar(x + off, vals, W,
               label=m, color=PAL[m], alpha=0.88,
               yerr=errs, capsize=3,
               error_kw={"linewidth":1.0, "ecolor":"#444"},
               edgecolor="white", linewidth=0.4, zorder=3)

    ylo = max(0, min(RES[ds][m][metric]
                     for ds in ds_list for m in MODEL_ORDER
                     if not np.isnan(RES[ds][m][metric])) - 0.06)
    ax.set_ylim(ylo, 1.02)
    ax.set_xticks(x)
    ax.set_xticklabels(ds_list, rotation=20, ha="right")
    ax.set_ylabel(metric)
    ax.set_title(f"5-fold CV {metric} — BLS variants across all datasets",
                 fontweight="bold")
    ax.legend(ncol=4, loc="lower right", framealpha=0.9)
    plt.tight_layout()
    path = OUT / fname
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show(); print(f"Saved: {path}")

fig_grouped_bars("AUC",   "fig1a_auc_grouped_bars.png")
fig_grouped_bars("GMean", "fig1b_gmean_grouped_bars.png")

## 9 · Figure 2 — Performance heatmaps

In [ ]:
def fig_heatmap(metric, fname, figsize=(13,5)):
    ds_list = list(RES.keys())
    data    = np.array([[RES[ds][m][metric] for m in MODEL_ORDER]
                         for ds in ds_list])
    fig, ax = plt.subplots(figsize=figsize)
    im  = ax.imshow(data.T, cmap="YlGn", aspect="auto",
                    vmin=max(0, np.nanmin(data)-0.05),
                    vmax=min(1, np.nanmax(data)+0.01))
    cbar = plt.colorbar(im, ax=ax, shrink=0.85)
    cbar.set_label(metric, fontsize=11)
    ax.set_xticks(range(len(ds_list)))
    ax.set_xticklabels(ds_list, rotation=22, ha="right")
    ax.set_yticks(range(len(MODEL_ORDER)))
    ax.set_yticklabels(MODEL_ORDER, fontsize=11)
    ax.set_title(f"{metric} — performance heatmap (all models × all datasets)",
                 fontweight="bold")
    for j, ds in enumerate(ds_list):
        col_vals = [RES[ds][m][metric] for m in MODEL_ORDER]
        best_v   = max(v for v in col_vals if not np.isnan(v))
        for i, (m, v) in enumerate(zip(MODEL_ORDER, col_vals)):
            txt = f"{v:.3f}" if not np.isnan(v) else "—"
            fw  = "bold" if v == best_v else "normal"
            ax.text(j, i, txt, ha="center", va="center",
                    fontsize=8.5, fontweight=fw,
                    color="black" if v < (np.nanmin(data)+np.nanmax(data))/2+0.1
                                  else "white")
    plt.tight_layout()
    path = OUT / fname
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show(); print(f"Saved: {path}")

fig_heatmap("AUC",   "fig2a_auc_heatmap.png")
fig_heatmap("GMean", "fig2b_gmean_heatmap.png")

## 10 · Figure 3 — Radar charts (all 4 metrics)

In [ ]:
def fig_radar_grid(fname, figsize=(14,10)):
    ds_list = list(DATASETS.keys())
    mets    = ["AUC","GMean","F1","MCC"]
    angles  = np.linspace(0, 2*np.pi, len(mets), endpoint=False).tolist()
    angles += angles[:1]

    ncols = 4; nrows = 2
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize,
                              subplot_kw=dict(polar=True))
    axes = axes.flatten()

    for ax_i, ds in enumerate(ds_list):
        ax = axes[ax_i]
        for mname in MODEL_ORDER:
            vals  = [RES[ds][mname][k] for k in mets] + [RES[ds][mname][mets[0]]]
            lw    = 2.5 if mname == "Con-BLS" else 1.4
            alpha = 0.15 if mname == "Con-BLS" else 0.04
            ax.plot(angles, vals, "o-", color=PAL[mname],
                    linewidth=lw, label=mname, markersize=4)
            ax.fill(angles, vals, alpha=alpha, color=PAL[mname])
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(mets, fontsize=9)
        ax.set_ylim(0.3, 1.05)
        ax.set_title(ds, pad=14, fontsize=10, fontweight="bold")

    # hide unused axes if any
    for ax_i in range(len(ds_list), len(axes)):
        axes[ax_i].set_visible(False)

    handles = [mpatches.Patch(color=PAL[m], label=m) for m in MODEL_ORDER]
    fig.legend(handles=handles, loc="lower center",
               ncol=4, fontsize=10, framealpha=0.9,
               bbox_to_anchor=(0.5, -0.01))
    fig.suptitle("Radar charts — AUC / G-Mean / F1 / MCC per dataset",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    path = OUT / fname
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show(); print(f"Saved: {path}")

fig_radar_grid("fig3_radar_all_datasets.png")

## 11 · Figure 4 — Ablation delta bars (improvement over BLS)

In [ ]:
def fig_delta_bars(fname, figsize=(14,5)):
    ds_list    = list(RES.keys())
    competitors = ["WBLS","CS-BLS","DKWBLS","Con-BLS"]
    n_met = len(METRICS); x = np.arange(len(ds_list))

    fig, axes = plt.subplots(1, n_met, figsize=figsize, sharey=False)
    W = 0.22; n = len(competitors)

    for ax, met in zip(axes, METRICS):
        for i, m in enumerate(competitors):
            deltas = [RES[ds][m][met] - RES[ds]["BLS"][met]
                      for ds in ds_list]
            off = (i - n/2 + 0.5) * W
            bars = ax.bar(x + off, deltas, W,
                          label=m, color=PAL[m], alpha=0.88,
                          edgecolor="white", linewidth=0.4, zorder=3)
        ax.axhline(0, color="#333", linewidth=0.8, linestyle="--")
        ax.set_xticks(x)
        ax.set_xticklabels(ds_list, rotation=22, ha="right", fontsize=8)
        ax.set_title(f"Δ {met}", fontweight="bold")
        ax.set_ylabel("delta vs BLS")

    handles = [mpatches.Patch(color=PAL[m], label=m) for m in competitors]
    fig.legend(handles=handles, loc="upper right",
               ncol=1, fontsize=10, framealpha=0.9)
    fig.suptitle("Ablation: improvement over standard BLS per dataset",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    path = OUT / fname
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show(); print(f"Saved: {path}")

fig_delta_bars("fig4_ablation_delta_bars.png")

## 12 · Figure 5 — G-Mean robustness vs imbalance ratio

In [ ]:
def run_ir_sweep(ir_list, n=1200, d=10, n_splits=3):
    """Evaluate all models as IR increases on synthetic binary data."""
    res = {m: [] for m in MODEL_ORDER}
    std = {m: [] for m in MODEL_ORDER}
    for ratio in tqdm(ir_list, desc="IR sweep"):
        rng = np.random.RandomState(SEED)
        nmi = max(4, int(n / (ratio + 1)))
        nmj = n - nmi
        X   = np.vstack([rng.randn(nmj, d),
                          rng.randn(nmi, d) + 2.8]).astype(np.float32)
        y   = np.array([0]*nmj + [1]*nmi)
        pm  = rng.permutation(len(y)); X, y = X[pm], y[pm]
        for mname in MODEL_ORDER:
            mn, sd = cross_validate(mname, X, y, n_splits=n_splits)
            res[mname].append(mn["GMean"])
            std[mname].append(sd["GMean"])
    return res, std

IR_LIST = [2, 5, 10, 20, 50, 100, 200]
print("Running IR robustness sweep…")
ir_res, ir_std = run_ir_sweep(IR_LIST)

def fig_ir_robustness(ir_list, ir_res, ir_std, fname, figsize=(9,5)):
    ls_map = {"BLS":"--", "WBLS":"-.", "CS-BLS":":", "DKWBLS":":", "Con-BLS":"-"}
    fig, ax = plt.subplots(figsize=figsize)
    for m in MODEL_ORDER:
        v = np.array(ir_res[m]); s = np.array(ir_std[m])
        lw = 2.5 if m == "Con-BLS" else 1.6
        ax.plot(ir_list, v, "o"+ls_map[m], color=PAL[m],
                lw=lw, label=m, markersize=5)
        ax.fill_between(ir_list, v-s, v+s, alpha=0.10, color=PAL[m])
    ax.set_xscale("log"); ax.set_xticks(ir_list)
    ax.set_xticklabels([f"{r}:1" for r in ir_list])
    ax.set_xlabel("Imbalance ratio")
    ax.set_ylabel("G-Mean")
    ax.set_title("G-Mean robustness as imbalance ratio increases",
                 fontweight="bold")
    ax.legend(); plt.tight_layout()
    path = OUT / fname
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show(); print(f"Saved: {path}")

fig_ir_robustness(IR_LIST, ir_res, ir_std, "fig5_ir_robustness.png")

## 13b · Figure 7 — t-SNE feature space: BLS vs Con-BLS

In [ ]:
from sklearn.manifold import TSNE

def fig_tsne_compare(ds_name, fname, nsub=400, figsize=(12,5)):
    X, y = DATASETS[ds_name]
    rng  = np.random.RandomState(SEED)
    if len(X) > nsub:
        idx = rng.choice(len(X), nsub, replace=False)
        Xs, ys = X[idx], y[idx]
    else:
        Xs, ys = X, y

    # BLS feature embeddings (random projections)
    bm = BLS(seed=SEED); bm.fit(Xs, ys)
    Zb = bm.feat_embed(Xs)

    # Con-BLS feature embeddings (contrastively trained)
    print(f"  Training Con-BLS for t-SNE ({ds_name})...")
    cm = ConBLS(epochs=50, bs=64, seed=SEED); cm.fit(Xs, ys)
    Zc = cm.feat_embed(Xs)

    perp = min(30, max(5, len(Xs) // 6))
    Eb   = TSNE(2, perplexity=perp, random_state=SEED,
                n_iter=500).fit_transform(Zb)
    Ec   = TSNE(2, perplexity=perp, random_state=SEED,
                n_iter=500).fit_transform(Zc)

    cls_col = {0: "#4C72B0", 1: "#C44E52"}
    fig, (a1, a2) = plt.subplots(1, 2, figsize=figsize)
    for ax, E, title in [(a1, Eb, "Standard BLS"),
                          (a2, Ec, "Con-BLS (proposed)")]:
        for ci in np.unique(ys):
            msk = ys == ci
            ax.scatter(E[msk,0], E[msk,1],
                       c=cls_col[ci],
                       marker="^" if ci > 0 else "o",
                       s=50 if ci > 0 else 15,
                       alpha=0.75,
                       label="Minority" if ci > 0 else "Majority",
                       edgecolors="none")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
        ax.legend(fontsize=9)
    fig.suptitle(f"Feature space: BLS vs Con-BLS — {ds_name}",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    path = OUT / fname
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show(); print(f"Saved: {path}")

fig_tsne_compare("Pima",  "fig7a_tsne_pima.png")
fig_tsne_compare("Glass", "fig7b_tsne_glass.png")

## 13 · Figure 6 — Box plots of fold-level performance

In [ ]:
def collect_fold_results(n_splits=N_FOLDS):
    """Re-run CV collecting per-fold results for box-plot visualisation."""
    fold_data = {m: {met: [] for met in METRICS} for m in MODEL_ORDER}
    for ds_name, (X, y) in DATASETS.items():
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                               random_state=SEED)
        for mname in MODEL_ORDER:
            for tr, te in skf.split(X, y):
                if len(np.unique(y[tr]))<2 or len(np.unique(y[te]))<2:
                    continue
                try:
                    m = make_model(mname); m.fit(X[tr], y[tr])
                    res = all_metrics(m, X[te], y[te])
                    for met in METRICS:
                        if not np.isnan(res[met]):
                            fold_data[mname][met].append(res[met])
                except Exception:
                    pass
    return fold_data

print("Collecting per-fold results for box plots…")
fold_data = collect_fold_results()

def fig_boxplots(fold_data, fname, figsize=(14,5)):
    fig, axes = plt.subplots(1, len(METRICS), figsize=figsize)
    colors    = [PAL[m] for m in MODEL_ORDER]
    for ax, met in zip(axes, METRICS):
        data = [fold_data[m][met] for m in MODEL_ORDER]
        bp   = ax.boxplot(data, patch_artist=True, notch=False,
                          medianprops={"color":"black","linewidth":2},
                          whiskerprops={"linewidth":1.2},
                          capprops={"linewidth":1.2},
                          flierprops={"marker":"o","markersize":3,
                                      "alpha":0.4})
        for patch, col in zip(bp["boxes"], colors):
            patch.set_facecolor(col); patch.set_alpha(0.75)
        ax.set_xticks(range(1, len(MODEL_ORDER)+1))
        ax.set_xticklabels(MODEL_ORDER, rotation=15)
        ax.set_title(met, fontweight="bold")
        ax.set_ylabel(met)
    fig.suptitle("Distribution of per-fold scores across all datasets",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    path = OUT / fname
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show(); print(f"Saved: {path}")

fig_boxplots(fold_data, "fig6_boxplots_fold_distribution.png")

## 14 · Wilcoxon significance tests & CSV export

In [ ]:
# ─── Wilcoxon signed-rank tests (DKWBLS vs each baseline) ────────────────────
def sig_tests(proposed="Con-BLS"):
    ds_list = list(RES.keys())
    comps   = [m for m in MODEL_ORDER if m != proposed]
    for met in METRICS:
        print(f"\nWilcoxon — {met}  ({proposed} vs …)")
        print(f"{'Baseline':<10} {proposed+' mean':>14} {'Base mean':>12}"
              f" {'p-value':>10}  Sig?")
        print("-"*54)
        ours = [RES[ds][proposed][met] for ds in ds_list]
        for comp in comps:
            base = [RES[ds][comp][met] for ds in ds_list]
            diff = np.array(ours) - np.array(base)
            if len(set(diff.round(6))) < 2:
                print(f"{comp:<10} {np.nanmean(ours):>14.4f}"
                      f" {np.nanmean(base):>12.4f}  {'n/a':>10}  —")
            else:
                _, p = wilcoxon(ours, base, alternative="greater")
                print(f"{comp:<10} {np.nanmean(ours):>14.4f}"
                      f" {np.nanmean(base):>12.4f}  {p:>10.4f}"
                      f"  {'Yes*' if p<0.05 else 'No'}")

sig_tests()  # Con-BLS vs all baselines

# ─── Export full results CSV ───────────────────────────────────────────────────
rows = []
for ds in RES:
    for m in MODEL_ORDER:
        row = {"Dataset": ds, "Model": m}
        for k in RES[ds][m]:
            row[k]          = RES[ds][m][k]
            row[k + "_std"] = STD[ds][m][k]
        rows.append(row)

df       = pd.DataFrame(rows)
csv_path = OUT / "bls_full_results.csv"
df.to_csv(csv_path, index=False)
print(f"\nCSV saved: {csv_path}")
print("\nAUC pivot:")
pivot = df.pivot_table(index="Dataset", columns="Model",
                       values="AUC")[MODEL_ORDER].round(4)
print(pivot.to_string())

## 15 · Package all outputs & download

In [ ]:
all_files = sorted(OUT.glob("*.png")) + sorted(OUT.glob("*.csv"))
print("Generated files:")
total_kb = 0
for f in all_files:
    kb = f.stat().st_size // 1024
    total_kb += kb
    print(f"  {f.name:<52} {kb:>5} KB")

zip_path = "BLS_Imbalanced_All_Outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in all_files:
        zf.write(f, f.name)

print(f"\nZIP: {zip_path}  ({total_kb} KB, {len(all_files)} files)")
try:
    from google.colab import files
    files.download(zip_path)
    print("Browser download triggered.")
except ImportError:
    print("Not in Colab — ZIP saved in working directory.")